# IMPORT LIBRAIRIES

In [21]:
import os, random
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
import time
from tqdm.auto import tqdm
import numpy as np




# Dataset

In [9]:
# Dataset

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff")

class RandomPatchSigmaMapDataset(Dataset):
    """
    Randomly sample patches from clean images and add Gaussian noise with random sigma.
    Returns:
      inp   (4,H,W) = noisy_rgb (3) + sigma_map (1)
      clean (3,H,W)
    """
    def __init__(self, clean_dir: str, patch: int = 30, sigma_min: float = 0.0, sigma_max: float = 50.0):
        self.paths = [
            os.path.join(clean_dir, f) for f in os.listdir(clean_dir)
            if f.lower().endswith(IMG_EXT)
        ]
        if not self.paths:
            raise ValueError(f"No images found in {clean_dir}")
        self.patch = patch
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max

    def __len__(self):
        # not important; training uses fixed steps per epoch
        return 1000000

    def _random_crop(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        if w < self.patch or h < self.patch:
            img = img.resize((max(w, self.patch), max(h, self.patch)))
            w, h = img.size
        x0 = random.randint(0, w - self.patch)
        y0 = random.randint(0, h - self.patch)
        return img.crop((x0, y0, x0 + self.patch, y0 + self.patch))

    def _augment(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = TF.hflip(img)
        if random.random() < 0.5:
            img = TF.vflip(img)
        k = random.randint(0, 3)
        if k:
            img = img.rotate(90 * k)
        return img

    def __getitem__(self, idx):
        path = random.choice(self.paths)
        img = Image.open(path).convert("RGB")
        img = self._random_crop(img)
        img = self._augment(img)

        clean = TF.to_tensor(img)  # (3,H,W) in [0,1]

        sigma = random.uniform(self.sigma_min, self.sigma_max)
        noise = torch.randn_like(clean) * (sigma / 255.0)
        noisy = (clean + noise).clamp(0.0, 1.0)

        sigma_map = torch.full((1, clean.shape[1], clean.shape[2]), sigma / 255.0)
        inp = torch.cat([noisy, sigma_map], dim=0)  # (4,H,W)

        return inp, clean


# ICRNN

In [10]:
# Model: IRCNN (7-layer dilated) + sigma map
class IRCNNSigmaMap(nn.Module):
    """
    Input:  (B,4,H,W) = noisy RGB + sigma map
    Output: (B,3,H,W) clean
    Residual learning: predict noise implicitly then subtract
    """
    def __init__(self, features: int = 64):
        super().__init__()
        dilations = [1, 2, 3, 4, 3, 2, 1]
        layers = []

        d = dilations[0]
        layers += [
            nn.Conv2d(4, features, 3, padding=d, dilation=d, bias=True),
            nn.ReLU(inplace=True),
        ]

        for d in dilations[1:-1]:
            layers += [
                nn.Conv2d(features, features, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ]

        d = dilations[-1]
        layers += [nn.Conv2d(features, 3, 3, padding=d, dilation=d, bias=True)]
        self.net = nn.Sequential(*layers)

    def forward(self, inp):
        pred_noise = self.net(inp)          # (B,3,H,W)
        noisy = inp[:, :3, :, :]
        clean = (noisy - pred_noise).clamp(0.0, 1.0)
        return clean

# Train

In [ ]:
# Train
def train(
    clean_dir=r"./BSDS300/images/train",
    out_dir="weights_ircnn_sigmap",
    patch=35,
    sigma_min=0.0,
    sigma_max=50.0,
    batch_size=8,
    steps_per_epoch=1000,
    max_epochs=10,
    lr0=1e-3,
    lr1=1e-4,
    plateau_epochs=5,
    log_every=50
):
    os.makedirs(out_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    ds = RandomPatchSigmaMapDataset(clean_dir, patch=patch, sigma_min=sigma_min, sigma_max=sigma_max)
    dl = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,              
        pin_memory=(device == "cuda"),
        drop_last=True
    )

    model = IRCNNSigmaMap().to(device).train()
    opt = Adam(model.parameters(), lr=lr0)
    loss_fn = nn.MSELoss()
    scaler = GradScaler(enabled=(device == "cuda"))

    best = float("inf")
    stagnant = 0
    using_lr1 = False

    global_step = 0

    for epoch in range(1, max_epochs + 1):
        running = 0.0
        start_t = time.time()

        pbar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch}/{max_epochs}", leave=True)
        for step, (inp, clean) in enumerate(dl):
            if step >= steps_per_epoch:
                break

            inp = inp.to(device, non_blocking=True)
            clean = clean.to(device, non_blocking=True)

            noisy = inp[:, :3, :, :]
            target_noise = noisy - clean

            with autocast(enabled=(device == "cuda")):
                pred_clean = model(inp)
                pred_noise = noisy - pred_clean
                loss = loss_fn(pred_noise, target_noise)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running += loss.item()
            global_step += 1

            # logs
            if (step + 1) % log_every == 0:
                avg_so_far = running / (step + 1)
                elapsed = time.time() - start_t
                it_s = (step + 1) / max(elapsed, 1e-9)
                pbar.set_postfix({
                    "loss": f"{avg_so_far:.5f}",
                    "lr": f"{opt.param_groups[0]['lr']:.1e}",
                    "it/s": f"{it_s:.2f}"
                })

            pbar.update(1)

        pbar.close()

        avg = running / max(1, steps_per_epoch)
        ckpt = os.path.join(out_dir, f"ircnn_sigmap_epoch{epoch:02d}.pth")
        torch.save({"model": model.state_dict(), "epoch": epoch}, ckpt)

        print(f"Epoch {epoch:02d} done | avg_loss={avg:.6f} | lr={opt.param_groups[0]['lr']:.1e}")

        # LR schedule + early stop like before
        if avg < best - 1e-7:
            best = avg
            stagnant = 0
        else:
            stagnant += 1

        if (not using_lr1) and stagnant >= plateau_epochs:
            for g in opt.param_groups:
                g["lr"] = lr1
            using_lr1 = True
            stagnant = 0
            print(f"Switch LR to {lr1}")

        if using_lr1 and stagnant >= plateau_epochs:
            print("Early stop: loss plateaued.")
            break

    final_path = os.path.join(out_dir, "ircnn_sigmap_final.pth")
    torch.save({"model": model.state_dict()}, final_path)
    print(f"Saved: {final_path}")


# Lance l'entraînement dans un notebook :
train()

/var/folders/vx/qd_bqlr943s7thdwg4zq7yq00000gn/T/ipykernel_61339/3437356284.py:36: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device == "cuda"))


device: cpu


Epoch 1/10:   0%|          | 0/1000 [00:00<?, ?it/s]/var/folders/vx/qd_bqlr943s7thdwg4zq7yq00000gn/T/ipykernel_61339/3437356284.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device == "cuda")):
Epoch 1/10: 100%|██████████| 1000/1000 [01:29<00:00, 11.19it/s, loss=0.00799, lr=1.0e-03, it/s=11.19]


Epoch 01 done | avg_loss=0.007990 | lr=1.0e-03


Epoch 2/10: 100%|██████████| 1000/1000 [01:30<00:00, 11.01it/s, loss=0.00268, lr=1.0e-03, it/s=11.01]


...

Epoch 8/10: 100%|██████████| 1000/1000 [01:32<00:00, 10.86it/s, loss=0.00134, lr=1.0e-03, it/s=10.86]


Epoch 08 done | avg_loss=0.001339 | lr=1.0e-03


Epoch 9/10: 100%|██████████| 1000/1000 [01:34<00:00, 10.62it/s, loss=0.00133, lr=1.0e-03, it/s=10.62]


Epoch 09 done | avg_loss=0.001326 | lr=1.0e-03


Epoch 10/10:  70%|███████   | 704/1000 [01:07<00:30,  9.79it/s, loss=0.00128, lr=1.0e-03, it/s=10.40]

In [32]:
import os, math
import torch
import torchvision.transforms.functional as TF
from PIL import Image
import torch.nn as nn

class IRCNNSigmaMap(nn.Module):
    def __init__(self, features: int = 64):
        super().__init__()
        dilations = [1, 2, 3, 4, 3, 2, 1]
        layers = []
        d = dilations[0]
        layers += [
            nn.Conv2d(4, features, 3, padding=d, dilation=d, bias=True),
            nn.ReLU(inplace=True),
        ]
        for d in dilations[1:-1]:
            layers += [
                nn.Conv2d(features, features, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ]
        d = dilations[-1]
        layers += [nn.Conv2d(features, 3, 3, padding=d, dilation=d, bias=True)]
        self.net = nn.Sequential(*layers)

    def forward(self, inp):
        pred_noise = self.net(inp)
        noisy = inp[:, :3, :, :]
        clean = (noisy - pred_noise).clamp(0.0, 1.0)
        return clean


def psnr_torch(x, y, eps=1e-8):
    # x,y in [0,1], shape (1,3,H,W)
    mse = torch.mean((x - y) ** 2).item()
    return 10.0 * math.log10(1.0 / (mse + eps))


@torch.no_grad()
def denoise_image_pil(model, img_pil, sigma, device):
    """
    img_pil: PIL RGB
    sigma: noise level in [0..50] (or whatever you trained), expressed in pixel space (0-255 scale)
    """
    y = TF.to_tensor(img_pil)  # (3,H,W) in [0,1]
    sigma_map = torch.full((1, y.shape[1], y.shape[2]), sigma / 255.0)
    inp = torch.cat([y, sigma_map], dim=0).unsqueeze(0).to(device)  # (1,4,H,W)
    out = model(inp).squeeze(0).cpu()  # (3,H,W)
    return TF.to_pil_image(out)


@torch.no_grad()
def test_mode_A_clean_to_noisy(
    clean_path,
    ckpt_path,
    out_dir="test_outputs",
    sigma=25.0,
    seed=0
):
    """
    Mode A: start from clean image -> add noise -> denoise -> compute PSNR vs clean.
    Saves: clean.png, noisy.png, denoised.png
    """
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = IRCNNSigmaMap().to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    clean_pil = Image.open(clean_path).convert("RGB")
    clean = TF.to_tensor(clean_pil).unsqueeze(0)  # (1,3,H,W)

    # create noisy
    g = torch.Generator().manual_seed(seed)
    noise = torch.randn(clean.shape, generator=g) * (sigma / 255.0)
    noisy = (clean + noise).clamp(0.0, 1.0)

    # denoise
    sigma_map = torch.full((1, 1, clean.shape[2], clean.shape[3]), sigma / 255.0)
    inp = torch.cat([noisy, sigma_map], dim=1).to(device)  # (1,4,H,W)
    den = model(inp).cpu()

    # PSNR
    psnr_noisy = psnr_torch(noisy, clean)
    psnr_den = psnr_torch(den, clean)

    # save
    TF.to_pil_image(clean.squeeze(0)).save(os.path.join(out_dir, "clean.png"))
    TF.to_pil_image(noisy.squeeze(0)).save(os.path.join(out_dir, f"noisy_sigma{int(sigma)}.png"))
    TF.to_pil_image(den.squeeze(0)).save(os.path.join(out_dir, f"denoised_sigma{int(sigma)}.png"))

    print("Saved to:", out_dir)
    print(f"PSNR noisy  : {psnr_noisy:.2f} dB")
    print(f"PSNR denoised: {psnr_den:.2f} dB")


@torch.no_grad()
def test_mode_B_real_noisy(
    noisy_path,
    ckpt_path,
    out_dir="test_outputs",
    sigma=25.0
):
    """
    Mode B: you have a noisy image (no GT) -> denoise and save.
    """
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = IRCNNSigmaMap().to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    noisy_pil = Image.open(noisy_path).convert("RGB")
    den_pil = denoise_image_pil(model, noisy_pil, sigma, device)

    base = os.path.splitext(os.path.basename(noisy_path))[0]
    den_pil.save(os.path.join(out_dir, f"{base}_denoised_sigma{int(sigma)}.png"))
    print("Saved to:", out_dir)


# VISUALIZE RESULTS OF DENOISING

In [33]:
test_mode_A_clean_to_noisy(
    clean_path=r"./BSDS300/images/test/42049.jpg",
    ckpt_path=r"./weights_ircnn_sigmap/ircnn_sigmap_final.pth",
    out_dir="test_outputs_100075",
    sigma=25.0
)

C:\Users\barra\AppData\Local\Temp\ipykernel_24976\1374041616.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location=device)


Saved to: test_outputs_100075
PSNR noisy  : 20.30 dB
PSNR denoised: 31.59 dB


# PSF (kernel) en OTF (fréquence) pour faire les étapes “blur/deblur” en FFT, puis alterner avec un CNN débruiteur dans une boucle HQS/PnP.

In [ ]:
def psf_to_otf(psf: torch.Tensor, H: int, W: int):
    """
    psf: (kh, kw) float tensor, sum=1
    returns OTF: (H, W) complex tensor
    """
    kh, kw = psf.shape
    pad = torch.zeros((H, W), device=psf.device, dtype=psf.dtype)
    pad[:kh, :kw] = psf

    pad = torch.roll(pad, shifts=(-(kh//2), -(kw//2)), dims=(0, 1))
    otf = torch.fft.fft2(pad)
    return otf


In [35]:
@torch.no_grad()
def ircnn_denoise_sigma(model, x01, sigma_pixels, device):
    """
    x01: (1,3,H,W) in [0,1]
    sigma_pixels: float in pixel scale (0..50)
    """
    H, W = x01.shape[-2], x01.shape[-1]
    sigma_map = torch.full((1, 1, H, W), float(sigma_pixels) / 255.0, device=device, dtype=x01.dtype)
    inp = torch.cat([x01.to(device), sigma_map], dim=1)
    out = model(inp).clamp(0.0, 1.0)
    return out


In [44]:
@torch.no_grad()
def deblur_pnp_hqs(
    model,
    y01,                  # (1,3,H,W) in [0,1], blurred (+ noise)
    psf,                  # (kh,kw) tensor, sum=1
    sigma_n_pixels=2.0,   # noise level in pixel units (0..)
    sigmas_pixels=None,   # list/tuple of denoiser sigmas (pixels), decreasing
    device="cuda"
):
    """
    Plug-and-Play HQS for non-blind deblurring with circular boundary (FFT).
    """
    model.eval()

    y = y01.to(device)
    B, C, H, W = y.shape
    assert B == 1 and C == 3

    if sigmas_pixels is None:
        # schedule typique: fort prior -> plus doux
        sigmas_pixels =[
    49.0, 46.0, 43.1, 40.4, 37.9, 35.5, 33.3, 31.2, 29.3, 27.5,
    25.8, 24.2, 22.8, 21.4, 20.1, 18.9, 17.8, 16.7, 15.7, 14.8,
    13.9, 13.1, 12.4, 11.7, 11.0, 10.4, 9.8, 9.2, 8.7, 8.2
]

    psf = psf.to(device, dtype=y.dtype)
    K = psf_to_otf(psf, H, W)                 # (H,W) complex
    Kc = torch.conj(K)
    K2 = (Kc * K).real                        # |K|^2 real

    # FFT de y (par canal)
    Y = torch.fft.fft2(y, dim=(-2, -1))       # (1,3,H,W) complex

    # init
    x = y.clone()
    z = x.clone()

    sigma_n = float(sigma_n_pixels) / 255.0
    sigma_n2 = sigma_n * sigma_n + 1e-12

    for s in sigmas_pixels:
        # relie sigma_denoise à beta : sigma_denoise ~= 1/sqrt(beta) (en [0,1])
        sigma_d = float(s) / 255.0
        beta = 1.0 / (sigma_d * sigma_d + 1e-12)

        # x-update (data fidelity) en FFT:
        # x = argmin (1/2σn^2)||Kx - y||^2 + (β/2)||x - z||^2
        # => X = (K*Y/σn^2 + β Z) / (|K|^2/σn^2 + β)
        Z = torch.fft.fft2(z, dim=(-2, -1))
        numerator = (Kc[None, None, :, :] * Y) / sigma_n2 + beta * Z
        denom = (K2[None, None, :, :] / sigma_n2 + beta)
        X = numerator / denom
        x = torch.fft.ifft2(X, dim=(-2, -1)).real.clamp(0.0, 1.0)

        # z-update via denoiser (Plug-and-Play proximal)
        z = ircnn_denoise_sigma(model, x, sigma_pixels=s, device=device)

    return z.clamp(0.0, 1.0)


In [ ]:
def gaussian_psf(ksize=15, sigma=3.0, device="cpu", dtype=torch.float32):
    ax = torch.arange(ksize, device=device, dtype=dtype) - (ksize - 1) / 2
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    psf = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    psf = psf / psf.sum()
    return psf

def blur_circular(x01, psf):
    """
    x01: (1,3,H,W)
    psf: (kh,kw)
    circular blur via FFT
    """
    device = x01.device
    B, C, H, W = x01.shape
    K = psf_to_otf(psf.to(device, dtype=x01.dtype), H, W)
    X = torch.fft.fft2(x01, dim=(-2, -1))
    Y = X * K[None, None, :, :]
    y = torch.fft.ifft2(Y, dim=(-2, -1)).real
    return y

def psnr_torch01(x, y, eps=1e-8):
    mse = torch.mean((x - y) ** 2).item()
    return 10.0 * math.log10(1.0 / (mse + eps))

@torch.no_grad()
def test_deblur_mode_A(
    clean_path,
    ckpt_path,
    out_dir="test_deblur",
    ksize=15,
    blur_sigma=10.0,
    sigma_n_pixels=2.0,
    seed=0
):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # model
    model = IRCNNSigmaMap().to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    clean_pil = Image.open(clean_path).convert("RGB")
    clean = TF.to_tensor(clean_pil).unsqueeze(0).to(device)

    psf = gaussian_psf(ksize=ksize, sigma=blur_sigma, device=device, dtype=clean.dtype)

    # blur
    blurred = blur_circular(clean, psf).clamp(0,1)

    # ajouter bruit
    g = torch.Generator(device=device).manual_seed(seed)
    noise = torch.randn(blurred.shape, device=device, dtype=blurred.dtype, generator=g) * (sigma_n_pixels / 255.0)

    y = (blurred + noise).clamp(0,1)

    # deblur PnP
    xhat = deblur_pnp_hqs(
        model,
        y01=y,
        psf=psf,
        sigma_n_pixels=sigma_n_pixels,
        sigmas_pixels=[
    49.0, 46.0, 43.1, 40.4, 37.9, 35.5, 33.3, 31.2, 29.3, 27.5,
    25.8, 24.2, 22.8, 21.4, 20.1, 18.9, 17.8, 16.7, 15.7, 14.8,
    13.9, 13.1, 12.4, 11.7, 11.0, 10.4, 9.8, 9.2, 8.7, 8.2
],
        device=device
    )

    # metrics
    psnr_blur = psnr_torch01(blurred, clean)
    psnr_y = psnr_torch01(y, clean)
    psnr_hat = psnr_torch01(xhat, clean)

    # save
    TF.to_pil_image(clean.squeeze(0).cpu()).save(os.path.join(out_dir, "clean.png"))
    TF.to_pil_image(blurred.squeeze(0).cpu()).save(os.path.join(out_dir, "blurred.png"))
    TF.to_pil_image(y.squeeze(0).cpu()).save(os.path.join(out_dir, f"blurred_noisy_sigN{int(sigma_n_pixels)}.png"))
    TF.to_pil_image(xhat.squeeze(0).cpu()).save(os.path.join(out_dir, "deblurred_pnp.png"))

    print("Saved to:", out_dir)
    print(f"PSNR blurred     : {psnr_blur:.2f} dB")
    print(f"PSNR blurred+noise: {psnr_y:.2f} dB")
    print(f"PSNR deblurred   : {psnr_hat:.2f} dB")


In [52]:
@torch.no_grad()
def deblur_real(
    noisy_blur_path,
    ckpt_path,
    out_dir="test_deblur",
    ksize=15,
    blur_sigma=3.0,
    sigma_n_pixels=2.0
):
    os.makedirs(out_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = IRCNNSigmaMap().to(device).eval()
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model"], strict=True)

    y_pil = Image.open(noisy_blur_path).convert("RGB")
    y = TF.to_tensor(y_pil).unsqueeze(0).to(device)

    psf = gaussian_psf(ksize=ksize, sigma=blur_sigma, device=device, dtype=y.dtype)

    xhat = deblur_pnp_hqs(
        model,
        y01=y,
        psf=psf,
        sigma_n_pixels=sigma_n_pixels,
        sigmas_pixels=[25, 18, 12, 8, 6, 4, 3, 2],
        device=device
    )

    base = os.path.splitext(os.path.basename(noisy_blur_path))[0]
    TF.to_pil_image(xhat.squeeze(0).cpu()).save(os.path.join(out_dir, f"{base}_deblurred_pnp.png"))
    print("Saved to:", out_dir)


In [ ]:
clean_dir = r"./BSDS300/images/test" 
candidates = []
for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
    candidates += glob.glob(os.path.join(clean_dir, "**", ext), recursive=True)

assert len(candidates) > 0, f"Aucune image trouvée dans {clean_dir}"
clean_path = random.choice(candidates)
print("Image choisie:", clean_path)

ckpt_path = r"weights_ircnn_sigmap/ircnn_sigmap_final.pth" 

# 3) Paramètres de flou + bruit pour le test
out_dir = "test_deblur"
ksize = 15
blur_sigma = 3.0        # flou gaussien (plus grand = plus flou)
sigma_n_pixels = 2.0    # bruit (0..)

test_deblur_mode_A(
    clean_path=clean_path,
    ckpt_path=ckpt_path,
    out_dir=out_dir,
    ksize=ksize,
    blur_sigma=blur_sigma,
    sigma_n_pixels=sigma_n_pixels,
    seed=0
)

print("Fichiers générés dans:", out_dir)
print(" - clean.png")
print(" - blurred.png")
print(" - blurred_noisy_sigN*.png")
print(" - deblurred_pnp.png")


Image choisie: ./BSDS300/images/test\306005.jpg


C:\Users\barra\AppData\Local\Temp\ipykernel_24976\3005879875.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(ckpt_path, map_location=device)


Saved to: test_deblur
PSNR blurred     : 23.30 dB
PSNR blurred+noise: 23.24 dB
PSNR deblurred   : 25.38 dB
Fichiers générés dans: test_deblur
 - clean.png
 - blurred.png
 - blurred_noisy_sigN*.png
 - deblurred_pnp.png
